# 45. Loss Function과 Class Imbalance

이 노트북은 segmentation에서 자주 쓰는 loss와 class imbalance 문제를 다룹니다.

이번 노트북의 목표는 다음과 같습니다.

- Cross Entropy loss의 입력 형태를 복습합니다.
- class imbalance가 pixel accuracy를 왜곡하는 이유를 확인합니다.
- class weight, Dice loss, Focal loss의 쓰임을 구분합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(2)

## 45-1. 불균형 mask 만들기

In [ ]:
b, c, h, w = 2, 3, 32, 32
target = torch.zeros((b, h, w), dtype=torch.long)
target[:, 6:12, 6:12] = 1
target[:, 20:24, 20:24] = 2

counts = torch.bincount(target.flatten(), minlength=c)
print("class pixel counts:", counts.tolist())
print("class ratios:", (counts / counts.sum()).round(decimals=4).tolist())

## 45-2. Pixel accuracy의 함정

배경 픽셀이 압도적으로 많으면 전부 background로 예측해도 정확도가 높게 나올 수 있습니다.

In [ ]:
all_background_pred = torch.zeros_like(target)
pixel_acc = (all_background_pred == target).float().mean()
print("all-background pixel accuracy:", round(float(pixel_acc), 4))

## 45-3. Weighted Cross Entropy

In [ ]:
logits = torch.randn((b, c, h, w), requires_grad=True)

plain_ce = nn.CrossEntropyLoss()(logits, target)

freq = counts.float().clamp_min(1)
weights = 1.0 / freq
weights = weights / weights.mean()
weighted_ce = nn.CrossEntropyLoss(weight=weights)(logits, target)

print("class weights:", weights.round(decimals=3).tolist())
print("plain CE:", round(float(plain_ce), 4))
print("weighted CE:", round(float(weighted_ce), 4))

## 45-4. Dice loss의 직관

In [ ]:
def dice_loss(logits, target, num_classes, eps=1e-6):
    probs = logits.softmax(dim=1)
    one_hot = F.one_hot(target, num_classes).permute(0, 3, 1, 2).float()
    dims = (0, 2, 3)
    intersection = (probs * one_hot).sum(dims)
    union = probs.sum(dims) + one_hot.sum(dims)
    dice = (2 * intersection + eps) / (union + eps)
    return 1 - dice.mean()


print("dice loss:", round(float(dice_loss(logits, target, c)), 4))

## 45-5. Loss 선택 기준

| 상황 | 먼저 고려할 loss |
|---|---|
| 기본 multi-class segmentation | Cross Entropy |
| class imbalance가 심함 | Weighted CE, Focal |
| 작은 객체나 영역 겹침이 중요함 | Dice, Lovasz 계열 |
| label noise가 많음 | CE 기반으로 안정성 확인 후 조합 |

처음부터 복잡한 loss를 쓰기보다, CE로 overfit small batch가 되는지 먼저 확인하는 편이 좋습니다.

## 정리

- segmentation에서는 배경 class가 많아 pixel accuracy가 쉽게 높아질 수 있습니다.
- class imbalance가 심하면 weighted CE, Dice, Focal loss를 검토합니다.
- 다음 노트북 `46_mIoU_Pixel_Accuracy_Confusion_Matrix_평가.ipynb`에서는 metric 계산을 더 구체적으로 다룹니다.